# Beyond Accuracy: Heart Disease Classification on Imbalanced Survey Data

Predicting self-reported heart disease from behavioral, medical, and demographic
features in the CDC BRFSS 2020 survey, and showing why **accuracy is the wrong
metric** when the classes are imbalanced.

**The short version:** four standard classifiers all score ~89% accuracy, but none
beats a model that just predicts "no disease" for everyone, and they detect only
1.5%-15.2% of actual heart disease cases. Two standard fixes (class weighting and
decision-threshold tuning) recover most of that lost sensitivity, raising detection
to roughly two-thirds of cases. The signal was there the whole time; the default
0.5 threshold was hiding it.

**Dataset:** a 3,087-row subset of the [Personal Key Indicators of Heart Disease](https://www.kaggle.com/datasets/kamilpytlak/personal-key-indicators-of-heart-disease)
dataset (derived from BRFSS 2020). Positive cases are 11.7% of records.


## 1. Setup

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn import svm
from sklearn.metrics import (
    confusion_matrix, accuracy_score, precision_score, recall_score,
    f1_score, balanced_accuracy_score, roc_auc_score, roc_curve,
    precision_recall_curve, average_precision_score, classification_report,
)

RANDOM_STATE = 0
plt.rcParams["figure.figsize"] = (8, 6)

## 2. Load the data

In [2]:
heart_df = pd.read_csv("data/heart_2020_cleaned_edited.csv").dropna()

# Encode the target: Yes -> 1, No -> 0
heart_df["HeartDisease"] = heart_df["HeartDisease"].map({"Yes": 1, "No": 0})

print("Shape:", heart_df.shape)
print()
print(heart_df["HeartDisease"].value_counts())
print()
prevalence = heart_df["HeartDisease"].mean()
print(f"Positive-class prevalence: {prevalence:.3f}  ({heart_df['HeartDisease'].sum()} of {len(heart_df)})")

Shape: (3087, 18)

HeartDisease
0    2726
1     361
Name: count, dtype: int64

Positive-class prevalence: 0.117  (361 of 3087)


In [3]:
heart_df.head()

,HeartDisease,BMI,Smoking,AlcoholDrinking,Stroke,PhysicalHealth,MentalHealth,DiffWalking,Sex,AgeCategory,Race,Diabetic,PhysicalActivity,GenHealth,SleepTime,Asthma,KidneyDisease,SkinCancer
0,0,16.60,Yes,No,No,3,30,No,Female,55-59,White,Yes,Yes,Very good,5,Yes,No,Yes
1,0,20.34,No,No,Yes,0,0,No,Female,80 or older,White,No,Yes,Very good,7,No,No,No
2,0,26.58,Yes,No,No,20,30,No,Male,65-69,White,Yes,Yes,Fair,8,Yes,No,No
3,0,24.21,No,No,No,0,0,No,Female,75-79,White,No,No,Good,6,No,No,Yes
4,0,23.71,No,No,No,28,0,Yes,Female,40-44,White,No,Yes,Very good,8,No,No,No


## 3. Preprocessing

Split predictors into continuous and categorical, one-hot encode the categorical
ones (dropping the first level of each to avoid redundancy), and recombine.


In [4]:
X = heart_df.drop(["HeartDisease"], axis=1)
y = heart_df["HeartDisease"]

categorical_cols = ["Race", "AgeCategory", "GenHealth", "Sex", "Smoking",
                    "AlcoholDrinking", "Stroke", "DiffWalking", "Diabetic",
                    "PhysicalActivity", "Asthma", "KidneyDisease", "SkinCancer"]

numerical = X.drop(categorical_cols, axis=1)
categorical = X[categorical_cols]

cat_numerical = pd.get_dummies(categorical, drop_first=True)
X = pd.concat([numerical, cat_numerical], axis=1)

print("Feature matrix shape after encoding:", X.shape)
X.head()

Feature matrix shape after encoding: (3087, 37)


,BMI,PhysicalHealth,MentalHealth,SleepTime,Race_Asian,Race_Black,Race_Hispanic,Race_Other,Race_White,AgeCategory_25-29,...,AlcoholDrinking_Yes,Stroke_Yes,DiffWalking_Yes,"Diabetic_No, borderline diabetes",Diabetic_Yes,Diabetic_Yes (during pregnancy),PhysicalActivity_Yes,Asthma_Yes,KidneyDisease_Yes,SkinCancer_Yes
0,16.60,3,30,5,False,False,False,False,True,False,...,False,False,False,False,True,False,True,True,False,True
1,20.34,0,0,7,False,False,False,False,True,False,...,False,True,False,False,False,False,True,False,False,False
2,26.58,20,30,8,False,False,False,False,True,False,...,False,False,False,False,True,False,True,True,False,False
3,24.21,0,0,6,False,False,False,False,True,False,...,False,False,False,False,False,False,False,False,False,True
4,23.71,28,0,8,False,False,False,False,True,False,...,False,False,True,False,False,False,True,False,False,False
